# Demo — Master Patient Index (MPI) Stewardship on the Ambiguous Band
In Veridian Health's `silver_patient_master` table, automated entity resolution links duplicate patient charts when `score >= 0.85` (`auto-link`) and rejects clear non-matches below `0.68`. Pairs scoring `0.68..0.84` land in the **`steward-review`** queue because automated rules cannot safely distinguish a patient who changed email providers from two different patients who share a name (**homonyms**).

## Step 0: Connect to your Cloud Lab

Run the cell below to configure your AWS Cloud Lab environment and seed the **Veridian Health Medallion Lakehouse** (`veridian_lakehouse` in AWS Glue, S3 `bronze/silver/gold` Parquet prefixes, and AWS Lake Formation enforcement) via `setup_aws.py`.

In [1]:
import os
os.environ["VERIDIAN_LF_LIVE"] = "1"
os.environ["VERIDIAN_AWS_REGION"] = "us-east-1"
# If your Vocareum AWS credentials are not already in the environment, paste them below:
# os.environ["AWS_ACCESS_KEY_ID"] = "..."
# os.environ["AWS_SECRET_ACCESS_KEY"] = "..."
# os.environ["AWS_SESSION_TOKEN"] = "..."

import boto3
import setup_aws
glue = boto3.client("glue", region_name=os.environ["VERIDIAN_AWS_REGION"])
try:
    glue.get_database(Name="veridian_lakehouse")
    print("Glue database 'veridian_lakehouse' is already seeded and ready.")
except glue.exceptions.EntityNotFoundException:
    setup_aws.main()

Glue database 'veridian_lakehouse' is already seeded and ready.


In [2]:
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from demo_toolkit import connect_lakehouse, lh_backend, adjudicate_mpi_matches

lh = connect_lakehouse()
mpi = lh.resolve_patient_mpi()
print(f"Backend: {lh_backend()} | Candidate pairs by band:")
print(mpi.candidates["band"].value_counts().to_string())

Backend: live | Candidate pairs by band:
band
steward-review    6
auto-link         2


## 1. Compare `auto-link` pairs vs. the `steward-review` queue

In [3]:
print("--- Auto-linked MPI pairs (score >= 0.85) ---")
display(mpi.candidates[mpi.candidates["band"] == "auto-link"])
print("\n--- Steward-review queue (0.68 <= score < 0.85) ---")
display(mpi.steward_queue)

--- Auto-linked MPI pairs (score >= 0.85) ---
   mpi_a       name_a                      email_a       phone_a jurisdiction_a  enrolled_a  mpi_b       name_b                      email_b       phone_b jurisdiction_b  enrolled_b  score       band
0   1048  Devon Solis     devon.solis@medinbox.net  +49660927833        EU-GDPR  2024-06-17   1241  Devon Solis     devon.solis@medinbox.net  +49660927833        EU-GDPR  2024-06-17    1.0  auto-link
1   1226   Mina Patel  mina.patel@veridianmail.org  +49865628550        EU-GDPR  2024-05-08   1242   Mina Patel  mina.patel@veridianmail.org  +49865628550        EU-GDPR  2024-05-08    1.0  auto-link

--- Steward-review queue (0.68 <= score < 0.85) ---
   mpi_a            name_a                         email_a       phone_a jurisdiction_a  enrolled_a  mpi_b            name_b                           email_b        phone_b jurisdiction_b  enrolled_b  score            band
0   1053  Leila Al-Mansoor  leila.al-mansoor@healthpost.io   +1479181391     

## 2. Adjudicate one true duplicate (`row 0`) and one homonym non-match (`row 4`)

- **Row 0 (`1053 <-> 1244`, Leila Al-Mansoor)**: trailing digit in email (`leila.al-mansoor6@healthpost.io`), but identical phone (`+1479181391`), jurisdiction (`CA-PIPEDA`), and enrollment date (`2024-03-30`) -> **`"accept"`**.
- **Row 4 (`1001 <-> 1247`, Freja Vargas)**: same name and email prefix, but completely different phone (`+1401213602` vs `+491769988776`), different regulatory jurisdiction (`CA-PIPEDA` vs `EU-GDPR`), and 3 years apart (`2024-11-23` vs `2021-09-10`) -> **`"reject"`** (merging two different patients' clinical charts is a severe clinical safety and HIPAA violation).

In [4]:
sample = adjudicate_mpi_matches(mpi, decisions={0: "accept", 4: "reject"}, survivor="lowest_mpi_id")
for k, v in sample.items():
    if k != "linked_pairs":
        print(f"{k:22}: {v}")
display(sample["linked_pairs"])

steward_queue_size    : 6
accepted_links        : 1
rejected_homonyms     : 1
survivorship_rule     : lowest mpi_id survives (Master Patient Index SoR)


   golden_mpi_id  retired_mpi_id                                       survivorship
0           1053            1244  lowest mpi_id survives (Master Patient Index SoR)
